> **💾 First: save your own copy** · This notebook opens from the course website, so Colab does not keep your changes. Before you do anything else, go to **File → Save a copy in Drive**. Colab opens your copy in a new tab: work in that tab and close this one. You find your copy later in Google Drive, in the folder *Colab Notebooks*.

> **⚠️ Important** · Please run the cell below to set up the environment.

In [ ]:
%%capture
#Keep cell

import os

# palettable is needed by CDAIML.py (colours of the neural-network drawing)
try:
  import palettable
except:
  !pip install palettable
  import palettable


if not(os.path.exists('snsdata.csv')):
  print('Downloading snsdata.csv\n')
  !wget https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/snsdata.csv

if not(os.path.exists('spotify-dataset.csv')):
  print('Downloading spotify-dataset.csv\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/spotify-dataset.csv

if not(os.path.exists('CDAIML.py')):
  print('Downloading CDAIML.py\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/refs/heads/main/CDAIML.py


from CDAIML import DrawNN
from CDAIML import display_youtube

![Cultural Data Analysis · Week 3: Structured Data and Machine Learning](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week3-banner.png)

# Machine Learning

*Lecture notebook* · Code-along: [goto4711.github.io/cda-teaching-tools/code-along/week3](https://goto4711.github.io/cda-teaching-tools/code-along/week3/) · Tools: [how a computer sees an image](https://goto4711.github.io/cda-teaching-tools/tools/images/), [neural network training](https://goto4711.github.io/cda-teaching-tools/tools/neural-net/)

---

> **Goals** · This notebook is about one question: **how does a machine learn from data?** We follow the same steps twice:
> - **Supervised learning:** a neural network learns from labelled examples to predict whether a song is danceable.
> - **Unsupervised learning:** k-means finds groups of teenagers with similar interests, without any labels.
>
> At every step, we also ask: **who decided?** Machines learn from data, but people choose the data, the labels, the features and the settings. Those choices shape what the machine learns.

Machine learning is much less scary than science fiction wants us to believe. Machines do not learn because they want to understand the meaning of life, but to complete a particular task, and they still learn mostly from human examples. *Supervised* learning uses data that people have labelled; *unsupervised* learning finds patterns in unlabelled data, which people then have to interpret. Both work on *structured data*: tables like a spreadsheet, with one row per observation and one column per *feature* (https://www.ibm.com/think/topics/structured-vs-unstructured-data).

Run the cell below to load today's libraries.

In [ ]:
#Keep cell
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### The data science process

Machine learning is a process of repeatable steps. Schutt and O'Neil (2013) give an excellent overview:

![alt text](https://github.com/goto4711/cdai/blob/main/process.png?raw=true "The Art of Data Science ")

We collect data, explore and clean it (often 80% of the work), decide on a question, train a model on part of the data, and evaluate it on the rest. Then we improve and repeat.

### How a machine sees similarity: feature spaces

Machines learn similarities as distances. Lantz (2013, *Machine Learning with R*) tastes 15 foods and records how crunchy and how sweet each is, on a scale from 1 to 10. In a *feature space* with one axis per feature, similar foods end up close together:

![alt text](https://github.com/goto4711/cdai/blob/main/lantz-2.png?raw=true "")

Now we taste a tomato for the first time. Is it a fruit or a vegetable? The machine decides by distance: which group is the tomato closest to?

![alt text](https://github.com/goto4711/cdai/blob/main/lantz-3.png?raw=true "")

> **🧭 Who decided?** · Someone chose to describe food by crunchiness and sweetness, on a scale from 1 to 10. With other features, such as colour or price, the tomato would land somewhere else.

---

# Supervised Machine Learning: Machine Dancing

Can a machine learn taste? For the philosopher Kant, a judgement of taste is subjective and yet demands that everyone agree with it (https://oll.libertyfund.org/titles/bernard-the-critique-of-judgement, §7). We try it with music: the machine should learn to predict whether a song is danceable. In the language of machine learning, this is a *classification* task with two classes.

## Step 1: Collect the data

Our data is a Kaggle copy of a dataset collected through Spotify's Web API: 114,000 tracks from 114 genres (https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset). Besides artist, title and genre, each track has *audio features* that Spotify's own algorithms computed from the music: danceability, energy, valence (how positive it sounds), acousticness, tempo and so on, mostly on a scale from 0 to 1. Spotify has since closed this part of its API, so nobody can collect these features today. The full list of columns: https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset.

Run the cell below to load the data.

In [ ]:
#Keep cell
music_df = pd.read_csv('spotify-dataset.csv')

Look at the data with `music_df.info()` and `music_df.head()`.

> **🧭 Who decided?** · Spotify's algorithms computed every audio feature, and we cannot see how. The dataset's creator chose 1,000 tracks per genre, and the genres themselves are Spotify's.

## Step 2: The label

`danceability` runs from 0.0 (least danceable) to 1.0 (most danceable). Plot its distribution with `sns.histplot(music_df['danceability'], bins=20)`.

For a classification, we need two classes: good or bad danceability. 0.6 splits the data roughly in half. `np.where` creates the new column `dance_label`: 1 for good, 0 for bad danceability. Run the cell below.

In [ ]:
#Keep cell
DANCEABILITY_THRESHOLD = 0.6 # Let's say songs with danceability >= 0.6 are "High Danceability"

music_df['dance_label'] = np.where(music_df['danceability'] >= DANCEABILITY_THRESHOLD, 1, 0)

Check the two classes with `music_df['dance_label'].value_counts()`.

What do the extremes sound like? The two cells below link to YouTube (click on the image: YouTube does not allow playing them inside the notebook).

In [ ]:
#Keep cell

least_danceable_example = "RWjxrp4C3xk"
display_youtube(least_danceable_example, video_title="Example of least danceable music")

In [ ]:
#Keep cell

most_danceable_example = "zVcMi4YCLWQ"
display_youtube(most_danceable_example, video_title="Example of most danceable music")

Which genres are the most danceable? Run the cell below.

In [ ]:
#Keep cell

plt.figure(figsize=(10, 6))
genre_counts = music_df['track_genre'].value_counts().nlargest(10)

avg_danceability_per_genre = music_df.groupby('track_genre')['danceability'].mean().sort_values(ascending=False).nlargest(10)
sns.barplot(x=avg_danceability_per_genre.values, y=avg_danceability_per_genre.index, hue = avg_danceability_per_genre.values, palette='magma', orient='h')
plt.title(f'Average Danceability for Top Genres')
plt.xlabel('Average Danceability Score')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

So, children's music is the most danceable? In the workshop, we look into such questions.

> **🧭 Who decided?** · The label is Spotify's danceability score, an algorithm's output, cut at 0.6. We chose the 0.6. The machine will learn to predict Spotify's score, not whether people dance.

## Step 3: Choose and prepare the features

The *features* are the columns the machine learns from. Run the cell below to choose them and copy them into `music_features_df`. `danceability` itself is left out: it is what we predict.

In [ ]:
#Keep cell

feature_cols = [
    'duration_ms', 'popularity', 'energy', 'key', # 'danceability' removed
    'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'tempo', 'time_signature', 'valence'
]

music_features_df = music_df[feature_cols].copy()

Remember the feature space: the machine compares songs by distance. If one feature is measured in milliseconds (`duration_ms`, up to millions) and another between 0 and 1, the large one dominates every distance. So we *normalise* all features to the same range from 0 to 1 with *min-max* normalisation: (value - minimum) / (maximum - minimum). Run the cell below to define it.

In [ ]:
#Keep cell

def normalize(x):
    return ((x - x.min()) / (x.max() - x.min()))

Apply `normalize` to every column and add the label back. Type in:
```
music_features_normalized_df = music_features_df.apply(normalize)
music_features_normalized_df['dance_label'] = music_df['dance_label']
music_features_normalized_df.head()
```

> **🧭 Who decided?** · We chose 13 features, including `popularity`, which measures how much a song is played on Spotify, not anything about the music. We also chose to give every feature the same weight.

## Step 4: Training and test data

We want to know how well the model predicts songs it has *not* seen. So we keep 20% of the songs aside as *test data* and train only on the other 80%. `train_test_split` shuffles the songs and splits them; `random_state=4711` makes the shuffle repeatable, so your results match this notebook. Run the cell below.

In [ ]:
#Keep cell
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(music_features_normalized_df, test_size=0.20, random_state=4711)
print(len(train_set), 'songs for training,', len(test_set), 'for testing')

scikit-learn wants the features (`X`) and the label (`y`) separately, as arrays (`.values`). Type in:
```
X_train = train_set.loc[:, train_set.columns != 'dance_label'].values
y_train = train_set['dance_label'].values

X_test = test_set.loc[:, test_set.columns != 'dance_label'].values
y_test = test_set['dance_label'].values
```

## Step 5: The model, a neural network

A neural network is loosely inspired by the brain: layers of units connected by *weights*. The input units take the features of a song, the output unit gives the prediction, and the *hidden* layers in between do the work.

![alt text](https://github.com/goto4711/cdai/blob/main/neural-networks-2.png?raw=true "")

The network learns by feedback, called *backpropagation*: it predicts the training songs, compares its predictions with the labels, and adjusts its weights to make fewer mistakes, again and again. Try it with the neural network tool: https://goto4711.github.io/cda-teaching-tools/tools/neural-net/.

scikit-learn's `MLPClassifier` (a multi-layer perceptron) is such a network. As in the code-along, there are three steps: make a model, fit it, predict. Type in:
```
from sklearn.neural_network import MLPClassifier
model = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=300, random_state=4711)
model.fit(X_train, y_train)
```
`hidden_layer_sizes=(20, 10)` means two hidden layers with 20 and 10 units; `max_iter=300` stops after at most 300 rounds of feedback. Training takes a minute or two.

What has the network learned? Only weights: numbers on the connections between the units. Run the cell below to draw them (orange connections are positive, blue ones negative).

In [ ]:
#Keep cell

network_structure = [13, 20, 10, 1]
weights = model.coefs_

vis = DrawNN(
    network_structure,
    weights_list=weights,
    title=f"Large Network {network_structure}",
    max_linewidth=2.5,
    linewidth_base=0.1,
    linewidth_scale_tier1=3.0,
    linewidth_scale_tier2=6.0,
    weight_text_show_threshold=0.7 # Show fewer weight texts for clarity
)
vis.draw()

> **🧭 Who decided?** · The size of the network (two layers, 20 and 10 units) and the number of rounds are *hyperparameters*: settings chosen by people before training, often by trying (https://en.wikipedia.org/wiki/Hyperparameter_(machine_learning)). Other choices give other networks.

## Step 6: Evaluate

Now predict the test songs, which the model has never seen, with `y_pred = model.predict(X_test)`.

A *confusion matrix* compares the predictions with the true labels. Run the cell below.

In [ ]:
#Keep cell
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_).plot()
plt.show()

The diagonal holds the correct predictions: bad songs predicted bad (top left) and good songs predicted good (bottom right). *Accuracy* is their share of all test songs. Type in `(cm[0, 0] + cm[1, 1]) / cm.sum()`.

About 79% of the predictions are correct. Is that good? Compare it with a machine that has learned nothing and always says 'not danceable'. Type in `(y_test == 0).mean()`.

Guessing 'not danceable' every time is right for about 55% of the songs, so the network has learned something. But remember the label: it has learned to predict Spotify's score.

> **🧭 Who decided?** · What counts as good enough? 79% may be impressive for a recommendation and unacceptable for a decision about people. And accuracy alone can hide which mistakes the model makes, as you will see with hate speech in Week 4.

---

# Unsupervised Machine Learning: Teenagers' Interests

A popular use of *clustering* is finding communities in social networks. Unlike supervised learning, there is no label: the machine groups the data by similarity, and people interpret the groups.

## Step 1: Collect the data

The data comes from Brett Lantz's book *Machine Learning with R* (2013, p. 279). It was compiled for sociological research on teenage identities: a random sample of 30,000 US high-school students with profiles on a social networking service in 2006, evenly spread over four graduation years (2006 to 2009). For each teenager, it records gender, age and number of friends. From the 500 most frequent words on the profiles, the researchers chose 36 words for five kinds of interest: extracurricular activities, fashion, religion, romance and antisocial behaviour. The data counts how often each word appears on each profile.

Run the cell below to load the data, and look at the first rows with `teens.head()`.

In [ ]:
#Keep cell
teens = pd.read_csv("snsdata.csv")

> **🧭 Who decided?** · The researchers chose the 36 words and the five kinds of interest. Whatever teenagers wrote about that is not on the list, the machine will never see.

## Step 2: Clean the data

Real-life data is messy: values are missing or wrong. Count the genders, including the missing ones, with `teens['gender'].value_counts(dropna=False)`.

2,724 teenagers have no gender recorded, and the data knows only 'F' and 'M'. For the analysis later, we need a column that is 1 for female and 0 for male, and stays empty where gender is missing. `map()` translates each value with a dictionary. Type in `teens['female'] = teens['gender'].map({'F': 1, 'M': 0})`.

Now the ages. Always look at the smallest and largest values: type in `teens['age'].describe()`.

Teenagers aged 3 or 106? As a rule of thumb, we say that high-school students are between 13 and 19, and we mark all other ages as missing. Run the cell below (as in the code-along).

In [ ]:
#Keep cell
too_young = teens['age'] < 13
too_old = teens['age'] >= 20
teens.loc[too_young | too_old, 'age'] = np.nan
teens['age'].isna().sum()

That makes 5,523 missing ages. Deleting all these teenagers would lose too much data, so we fill in an estimate. The average of everyone would do, but we can do better: teenagers who graduate earlier are older. Check the average age per graduation year with `teens.groupby('gradyear')['age'].mean()`.

Now fill each missing age with the average of its graduation year. `transform('mean')` gives every row the average of its own group, so the averages line up with the ages. Type in:
```
year_mean = teens.groupby('gradyear')['age'].transform('mean')
teens['age'] = teens['age'].fillna(year_mean)
teens['age'].describe()
```

> **🧭 Who decided?** · We decided that teenagers are between 13 and 19, and that a missing age is the average of the graduation year. Gender was recorded in two categories only. Every filled-in age is our assumption, not the teenager's answer.

## Step 3: Choose and prepare the features

We cluster by interests only: the 36 word columns, from basketball to drugs. Then we scale them, as with the music: k-means compares teenagers by distance, and words that are used a lot would otherwise dominate. This time we use the *z-score*: how many standard deviations a value lies above or below the average of its column (https://www.statisticshowto.com/probability-and-statistics/z-score/). Type in:
```
from scipy.stats import zscore
interests = teens.loc[:, 'basketball':'drugs']
interests_z = interests.apply(zscore)
interests_z.head()
```

## Step 4: The model, k-means

k-means (MacQueen, 1967; https://en.wikipedia.org/wiki/K-means_clustering) is old but still widely used, because it is fast and simple. You stepped through it by hand in the code-along:
1. place k centres at random;
2. assign every observation to its nearest centre;
3. move each centre to the middle of its observations, and repeat until nothing changes.

We have to tell k-means how many clusters to find. Run the cell below to choose `k = 5`.

In [ ]:
#Keep cell

k = 5

Make a `KMeans` model called `teen_clusters` with `n_clusters = k` and `random_state=4711`, and fit it to `interests_z`. Then count the teenagers in each cluster with `np.bincount(teen_clusters.labels_)`. Type in:
```
from sklearn.cluster import KMeans
teen_clusters = KMeans(n_clusters = k, random_state=4711)
teen_clusters.fit(interests_z)
np.bincount(teen_clusters.labels_)
```

> **🧭 Who decided?** · We chose k = 5, following Lantz. With another k, k-means finds other groups, and it always finds k groups, whether they exist or not. In the workshop, you will try to find a good k.

## Step 5: Interpret the clusters

The *centre* of each cluster is the average of its teenagers' (scaled) interests. Run the cell below: it shows the five interests that score highest in each cluster.

In [ ]:
#Keep cell
interests_centroids = pd.DataFrame(teen_clusters.cluster_centers_, columns=interests_z.columns)

for c in range(k):
    top = interests_centroids.loc[c].sort_values(ascending=False).head(5)
    print('Cluster', c, ':', list(top.index))

Lantz (2013, p. 288) named his clusters like this. Do your clusters match? Do the names make sense to you? Do you remember all those teenage films?

![alt text](https://github.com/goto4711/cdai/blob/main/teen-clusters.png?raw=true "")

Now we can ask who is in each cluster. Add the cluster of each teenager to `teens` with `teens['cluster'] = teen_clusters.labels_`, and compute the average age, share of girls and number of friends per cluster with `teens.groupby('cluster')[['age', 'female', 'friends']].mean()`.

The clusters hardly differ in age. Most users are girls (81% of those whose gender is known), so they dominate every cluster, but more in some than in others. The number of friends differs most: interests seem to go together with how social a teenager is online. Note that `female` is empty where gender is missing, so its mean is the share of girls among those whose gender is known.

> **🧭 Who decided?** · The machine only found groups of numbers. We named them, and the names, such as Lantz's 'princesses' or 'criminals', are ours. They also carry our ideas about teenagers.

---

## The steps, and who decided

| Step | Danceable songs (supervised) | Teenagers' interests (unsupervised) |
|---|---|---|
| 1. Collect | Spotify's API; features computed by Spotify | 30,000 profiles; 36 words chosen by researchers |
| 2. Label or clean | danceability ≥ 0.6, our threshold | ages 13–19 only; missing ages filled by graduation year |
| 3. Features | 13 features we chose, normalised to 0–1 | 36 interests, scaled with z-scores |
| 4. Model | a neural network: 2 layers, 20 and 10 units | k-means with k = 5 |
| 5. Evaluate or interpret | 79% accuracy against 55% for guessing | five groups, named by us |

The same steps come back in every machine-learning project, and in your group project. At each of them, people make choices that the machine then learns from.

### Check an agent

Ask Gemini to "cluster the teenagers in teens by their interests with k-means into five clusters and describe the clusters". Run its code in the cell below. Does it scale the interests? Which columns does it use? Does it do anything about the impossible ages? Do you get the same clusters? You can only check this because you did it yourself first.

In [ ]:
# Paste Gemini's code here, run it and compare